# Kuukausittaisten autovakuutuksen korvausmäärien kausiennuste PROC FORECAST -menetelmällä

## Yhteenveto

Vakuutusmatemaattinen varausryhmä tarvitsee 12 kuukauden ennakkokatsauksen kuukausittaisiin autovakuutuksen korvausmääriin, jotka osoittavat tasaista salkun kasvua sekä selvää talvisään aiheuttamaa nousua. Tämä muistikirja luo viiden vuoden synteettiset kuukausittaiset korvausmäärät (60 kuukautta, tammikuu 2020 - joulukuu 2024, vaihdellen noin 1 460:sta 2 780:een korvaukseen), ja käyttää sitten **PROC FORECAST** -menetelmää sovittaakseen askelittaisen autoregressiivisen perustason mallin ja multiplikatiivisen Holt-Winters-kausimallin. Kumpikin malli tuottaa 12 kuukauden pistennusteet 95 %:n luottamusväleillä kapasiteetti- ja varaussuunnittelua varten. Kausivaihteleva Holt-Winters-malli ennustaa voimakkaimman kysynnän yhden - kahden kuukauden päähän (noin 2 780-2 900 korvausta), tasoittuen pohjalukemaan noin askeleen yhdeksän kohdalla (noin 2 200), kun taas autoregressiivinen perustaso ennustaa tasaisemman laskun; molempien luottamusvälit levenevät ennustehorisontin myötä.

## Tietolähteet

| Tietojoukko | Rivit | Taso | Avainmuuttujat | Kuvaus |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Yksi rivi per kalenterikuukausi (tammikuu 2020 - joulukuu 2024) | `date` (SAS-päivämäärä, `MONYY7.`), `claim_count` (numeerinen) | Synteettiset kuukausittaiset autovakuutuksen korvausmäärät, jotka on rakennettu lineaarisesta kasvutrendistä (~9 korvausta/kk), sinimuotoisesta vuotuisesta syklistä, additiivisesta joulu-/tammi-/helmikuun talvisään aiheuttamasta piikistä ja Gaussin kohinasta (`rand('normal')`). Siemenluku `20240531` tekee siitä toistettavan. |

# Kuukausittaisten autovakuutuksen korvausmäärien kausiennuste

Henkilöasiakasvakuuttajan varaus- ja kapasiteettisuunnitteluryhmät tarvitsevat ennakkokatsauksen siihen, kuinka monta **autovahinkoa** ilmoitetaan kuukausittain. Tämän salkun korvausmäärä kasvaa tasaisesti salkun laajentuessa, ja se piikittää joka talvi, kun jää, lumi ja vähentynyt päivänvalo lisäävät kolari- ja lasivahinkoja.

Tämä muistikirja käy läpi täydellisen `PROC FORECAST`-työnkulun:

1. Luo realistinen, täysin synteettinen kuukausittainen korvausmääräsarja.
2. Sovita **askelittainen autoregressiivinen (STEPAR)** perustaso, joka vangitsee trendin ja autokorrelaation.
3. Sovita **multiplikatiivinen Holt-Winters (WINTERS)** -malli, joka mallintaa nimenomaisesti 12 kuukauden kausisyklin.
4. Vertaa kahta mallia ja tulkitse eteenpäin suuntautuva ennuste ja luottamusväli.

Kaikki toimii sisäisen synteettisen datan varassa - ei ulkoisia tiedostoja eikä verkkoyhteyttä.

## Vaihe 1 - Synteettisen korvaussarjan luonti

Alla oleva DATA-askel rakentaa **60 kuukausittaista havaintoa** (tammikuusta 2020 joulukuuhun 2024). Jokaiselle kuukaudelle yhdistämme neljä komponenttia, jotka jäljittelevät todellista autovakuutussalkkua:

- **Trendi** - noin 1 800 korvauksen perustaso, joka kasvaa noin 9:llä kuukaudessa vakuutuskannan kasvaessa.
- **Vuosisykli** - sini-/kosinitermi, joka antaa sulavan kausittaisen aallon.
- **Talvipiikki** - additiivinen piikki joulu-, tammi- ja helmikuussa.
- **Kohina** - `rand('normal', 0, 90)` realistista kuukaudesta toiseen vaihtelua varten.

`call streaminit()` kiinnittää satunnaislukuvirran, jotta muistikirja on toistettavissa. `date`-muuttuja on todellinen SAS-päivämäärä, joka on rakennettu `INTNX`-funktiolla ja muotoiltu `MONYY7.`-muodossa, ja `ID date INTERVAL=MONTH` -lause nimeää sen aikatunnisteeksi. Alla tulostetut ensimmäiset 14 riviä osuvat välille noin 1 460-2 450 korvausta.

In [1]:
TIEDOT claims;
    CALL streaminit(20240531);
    TEE i = 0 ASTI 59;
        /* Rakenna todellinen SAS-kuukausipäivämäärä, jotta INTERVAL=MONTH kohdistuu oikein */
        date = intnx('month', '01JAN2020'd, i);
        MUOTO date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = tammikuu ... 12 = joulukuu */

        trend  = 1800 + 9 * i;               /* kasvava vakuutuskannan perustaso */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx SISÄLLÄ (12, 1, 2));   /* jää-/lumipiikki  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        JOS claim_count < 0 NIIN claim_count = 0;
        TULOSTE;
    LOPPU;
    SÄILYTÄ date claim_count;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=claims(obs=14) NIMIKE;
    NIMIKE date = "Kuukausi" claim_count = "Ilmoitetut korvaukset";
    OTSIKKO "Ensimmäiset 14 kuukautta synteettisiä autovakuutuksen korvausmääriä";
SUORITA;

                          Ensimmäiset 14 kuukautta synteettisiä autovakuutuksen korvausmääriä                           

  Obs  Kuukausi  Ilmoitetut korvaukset
    1     21915                   2178
    2     21946                   2281
    3     21975                   2252
    4     22006                   1974
    5     22036                   2067
    6     22067                   1805
    7     22097                   1697
    8     22128                   1619
    9     22159                   1462
   10     22189                   1688
   11     22220                   1713
   12     22250                   2008
   13     22281                   2116
   14     22312                   2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Vaihe 2 - Askelittainen autoregressiivinen perustaso (METHOD=STEPAR)

`METHOD=STEPAR` on oletusarvo. Se sovittaa ensin aikatrendimallin - tässä `TREND=2` lineaarista trendiä varten - ja soveltaa sitten **askelittaista autoregressiota** residuaaleihin, tuoden mukaan ja säilyttäen AR-viiveitä merkitsevyyden perusteella. `AR=4` rajoittaa ehdokasautoregressiivisen asteen neljään viiveeseen, mikä riittää hyvin kuukausittaiselle sarjalle, jolla on lyhyen aikavälin momentumia.

Tässä käytetyt keskeiset asetukset:

- `LEAD=12` - ennusta 12 kuukautta datan yli.
- `ALPHA=0.05` - 95 %:n luottamusvälit.
- `OUTFULL` - pinoa historialliset `ACTUAL`-rivit yhteen ennustehorisontin rivien kanssa `OUT=`-tietojoukossa (pelkkien ennusterivien sijaan).
- `OUTLIMIT` - lisää alempi/ylempi luottamusväli-**sarakkeet** (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` - nimeää aikatunnisteen ja ilmoittaa sarjan olevan kuukausittainen.

In [2]:
PROSEDUURI forecast TIEDOT=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    MUUTTUJA claim_count;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=fc_stepar(obs=10) NIMIKE;
    OTSIKKO "STEPAR-tuloste: toteutuneet, sovitetut ja ennusterivit";
SUORITA;

                          Ensimmäiset 14 kuukautta synteettisiä autovakuutuksen korvausmääriä                           

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                 STEPAR-tuloste: toteutuneet, sovitetut ja ennusterivit                                 

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### OUT=-tietojoukon lukeminen

`OUT=`-tietojoukko pinoaa rivit `_TYPE_`-sarakkeen mukaan ja kantaa luottamusvälit sivu**sarakkeina**:

| Elementti | Laji | Merkitys |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | rivi | Havaittu `claim_count` jokaiselle 60 historialliselle kuukaudelle. |
| `_TYPE_ = 'FORECAST'` | rivi | 12 pisteennustetta ennustehorisontille. |
| `L95_claim_count` / `U95_claim_count` | sarake | Alempi / ylempi 95 %:n luottamusväli, täytetty `FORECAST`-riveillä (puuttuu `ACTUAL`-riveiltä). Numeerinen taso heijastaa `ALPHA=`-asetusta. |

Yllä tulostettu `OUT=` sisältää siis 72 riviä: 60 `ACTUAL`-historiariviä, joita seuraa 12 `FORECAST`-horisonttiriviä. Ensimmäiset kymmenen yllä näytettyä riviä ovat kaikki `ACTUAL`, ja luottamusvälisarakkeet puuttuvat, koska rajat liittyvät vain ennusteriveihin.

> **Moottorihuomautus.** SAS:n `OUTFULL` kirjoittaa myös otoksen sisäisen yhden askeleen `FORECAST`-rivin jokaiselle historialliselle kuukaudelle, ja `OUTRESID` lisää `_TYPE_='RESIDUAL'`-rivit. Jennerin PROC FORECAST ei vielä tuota näitä otoksen sisäisiä sovitus-/residuaalirivejä (seurataan puutetestinä `400979`), joten tämä muistikirja lukee vain `ACTUAL`-historian ja eteenpäin suuntautuvan `FORECAST`-horisontin.

## Vaihe 3 - Kausivaihteleva Holt-Winters-malli (METHOD=WINTERS)

STEPAR-malli vangitsee trendin ja lyhyen aikavälin autokorrelaation, mutta ei mallinna nimenomaisesti toistuvaa joulukuu-helmikuu-nousua. Sarjalle, jolla on selkeä vuotuinen rytmi, **multiplikatiivinen Holt-Winters** on parempi työkalu.

`METHOD=WINTERS` hajottaa sarjan tasoksi, lineaariseksi trendiksi (`TREND=2`) ja **multiplikatiiviseksi kausikertoimeksi**. `SEASONS=12` ilmoittaa 12 jakson (kuukausittaisen) kausisyklin. Pyydämme jälleen 12 kuukauden `LEAD`-ennusteen 95 %:n rajoin ja `OUTFULL OUTLIMIT`, jotta tulos linjautuu STEPAR-ajon kanssa.

Koska moottori etenee ennusteen `ID`-arvoa yhdellä yksiköllä per askel (sen sijaan että kunnioittaisi `INTERVAL=MONTH`-asetusta horisontin päivämäärille - puutetesti `400979`), alla oleva solu tarkastelee ennustetta **kuukausina eteenpäin (askel 1-12)** sen sijaan, että luottaisi tulostettuihin kalenteripäivämääriin.

In [3]:
PROSEDUURI forecast TIEDOT=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    MUUTTUJA claim_count;
SUORITA;

/* Säilytä 12 kuukauden eteenpäin suuntautuva horisontti ja indeksoi se askeleen mukaan (1..12). */
TIEDOT horizon;
    ASETA fc_winters;
    PIDÄ months_ahead 0;
    JOS _type_ = 'FORECAST';
    months_ahead + 1;
    SÄILYTÄ months_ahead claim_count l95_claim_count u95_claim_count;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=horizon NIMIKE;
    NIMIKE months_ahead     = "Kuukausia eteenpäin"
          claim_count       = "Ennustetut korvaukset"
          l95_claim_count   = "Alaraja 95 %"
          u95_claim_count   = "Yläraja 95 %";
    OTSIKKO "Holt-Winters 12 kuukauden eteenpäin suuntautuva ennuste (askelittain)";
SUORITA;

                                 STEPAR-tuloste: toteutuneet, sovitetut ja ennusterivit                                 

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                         Holt-Winters 12 kuukauden eteenpäin suuntautuva ennuste (askelittain)                          

  Obs   Kuukausia eteenpäin  Ennustetut korvaukset  Alaraja 95 %   Yläraja 95 %
    1                     1             2783.07951   2577.844742    2988.314278
    2                     2            2897.355557   2607.109764    3187.601349
    3                     3            2805.272075   2449.795029     3160.74912
    4                     4            2664.498049   2254.028514    3074.967585
    5                     5            2628.810136   2169.891244    3087.729029
    6                     6             2548.39319   2045.672732    3051.113649
    7                     7            2391.649756     1848.6496    2934.649912
    8  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Vaihe 4 - Kahden mallin vertailu rinnakkain

Selkein tapa arvioida, ansaitseeko kausimalli paikkansa, on asettaa sen 12 kuukauden ennuste rinnakkain STEPAR-perustason kanssa, askel askeleelta. Poimimme `FORECAST`-rivit molemmista `OUT=`-tietojoukoista, indeksoimme kunkin kuukausia-eteenpäin-arvon mukaan ja yhdistämme ne, jotta ero näkyy selvästi.

Kaksi menetelmää ovat samaa mieltä yleisestä tasosta, mutta eri mieltä **muodosta**: Holt-Winters kuljettaa vuotuisen rytmin horisonttiin (korkeampi taso ennustehorisontin alussa, joka tasoittuu keskivaiheen pohjalukemaan ja nousee jälleen), kun taas STEPAR - joka mallintaa kausivaihtelua vain epäsuorasti AR-viiveiden kautta - vaimenee tasaisemmin kohti sarjan keskiarvoa. Näiden kahden välinen ero jokaisessa askeleessa on se kausisignaali, jonka STEPAR jättää huomiotta.

> SAS ajaisi normaalisti tämän riittävyystarkistuksen `OUTRESID`-yhden askeleen residuaaleilla (`_TYPE_='RESIDUAL'`). Jenner ei vielä täytä näitä rivejä (puutetesti `400979`), joten vertaamme kahta eteenpäin suuntautuvaa ennustetta suoraan sen sijaan - diagnostiikka, joka käyttää vain sitä tulostetta, jonka moottori todella tuottaa.

In [4]:
/* STEPAR-horisontti, indeksoitu kuukausia-eteenpäin-arvon mukaan. */
TIEDOT stepar_h;
    ASETA fc_stepar;
    PIDÄ months_ahead 0;
    JOS _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    SÄILYTÄ months_ahead stepar;
SUORITA;

/* WINTERS-horisontti, indeksoitu kuukausia-eteenpäin-arvon mukaan. */
TIEDOT winters_h;
    ASETA fc_winters;
    PIDÄ months_ahead 0;
    JOS _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    SÄILYTÄ months_ahead winters;
SUORITA;

/* Yhdistä molemmat ja näytä kausiero, jonka STEPAR jättää huomiotta. */
TIEDOT COMPARE;
    YHDISTÄ stepar_h winters_h;
    MUKAAN months_ahead;
    seasonal_gap = winters - stepar;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=COMPARE NIMIKE;
    NIMIKE months_ahead = "Kuukausia eteenpäin"
          stepar        = "STEPAR-ennuste"
          winters       = "Wintersin ennuste"
          seasonal_gap  = "Winters - STEPAR";
    MUOTO stepar winters seasonal_gap 8.0;
    OTSIKKO "STEPAR vs. Holt-Winters: 12 kuukauden ennustevertailu";
SUORITA;

                                 STEPAR vs. Holt-Winters: 12 kuukauden ennustevertailu                                  

  Obs   Kuukausia eteenpäin  STEPAR-ennuste  Wintersin ennuste  Winters - STEPAR
    1                     1            2619               2783               164
    2                     2            2537               2897               361
    3                     3            2384               2805               421
    4                     4            2214               2664               450
    5                     5            2089               2629               540
    6                     6            2010               2548               539
    7                     7            1982               2392               410
    8                     8            1995               2240               245
    9                     9            2031               2197               166
   10                    10            2075               2354      


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Vaihe 5 - Ennusta jokainen liiketoimintalinja kerralla (BY-käsittely)

Todelliset varausajot kattavat useita tuotteita. Kun data on lajiteltu `line_of_business`-muuttujan mukaan, `BY`-lause saa `PROC FORECAST`:n sovittamaan **riippumattoman kausimallin jokaiselle ryhmälle** yhdessä kutsussa. Tässä synteettisoimme autolinjan (korkeampi perusvolyymi) ja kotilinjan (matalampi perusvolyymi) ja ennustamme molempia kuusi kuukautta eteenpäin. `OUTALL` kirjoittaa koko rivijoukon - `ACTUAL`-historian ja `FORECAST`-horisontin - yhdessä rajasarakkeiden kanssa jokaiselle ryhmälle, ja pidämme alla olevaan taulukkoon vain `FORECAST`-rivit.

In [5]:
TIEDOT multi_book;
    CALL streaminit(20240531);
    PITUUS line_of_business $6;
    TEE lob = 1 ASTI 2;
        JOS lob = 1 NIIN line_of_business = 'Auto';
        MUUTEN            line_of_business = 'Koti';
        TEE i = 0 ASTI 47;
            date = intnx('month', '01JAN2021'd, i);
            MUOTO date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi SISÄLLÄ (12, 1, 2))
                + rand('normal', 0, 70));
            TULOSTE;
        LOPPU;
    LOPPU;
    SÄILYTÄ line_of_business date claim_count;
SUORITA;

PROSEDUURI LAJITTELE TIEDOT=multi_book;
    MUKAAN line_of_business date;
SUORITA;

PROSEDUURI forecast TIEDOT=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    MUKAAN line_of_business;
    id date interval=month;
    MUUTTUJA claim_count;
SUORITA;

PROSEDUURI TULOSTA TIEDOT=fc_by(obs=12) NIMIKE;
    MISSÄ _type_ = 'FORECAST';
    NIMIKE line_of_business = "Liiketoimintalinja" date = "Kuukausi"
          claim_count = "Korvausmäärä" l95_claim_count = "Alaraja 95 %"
          u95_claim_count = "Yläraja 95 %";
    OTSIKKO "6 kuukauden ennusteet liiketoimintalinjoittain";
SUORITA;

                                 STEPAR vs. Holt-Winters: 12 kuukauden ennustevertailu                                  


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Koti

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                     6 kuukauden ennusteet liiketoimintalinjoittain                                     

  Obs  Liiketoimintalinja  Kuukausi    _TYPE_     Korvausmäärä  Alaraja 95 %   Yläraja 95 %  RESIDUAL_CLAIM_COUNT
    1  Auto                   23742  FORECAST      2524.596717   2359.095846    2690.097589                     .
    2  Auto                   23773  FORECAST      2604.418724   2370.365147      2838.4723                     .
    3  Auto                   23801  FORECAST      2576.092313   2289.436395     2862.74823                     .
    4  Auto       


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Koti
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Tulosten tulkinta

**Mitä mallit kertovat varausryhmälle:**

- **STEPAR** toistaa nousevan trendin ja lyhyen aikavälin momentumin, mutta sen 12 kuukauden ennuste vaimenee tasaisesti noin 2 620:sta (askel 1) noin 1 980:een horisontin puolivälissä ennen kuin ajautuu takaisin noin 2 140:een - se ei toista terävää talvipiikkiä, koska se käsittelee kausivaihtelua vain autoregressiivisten viiveiden kautta. Se on kohtuullinen nopea perustaso.
- **WINTERS** ja `SEASONS=12` kuljettavat vuotuisen rytmin suoraan multiplikatiivisten kausikertoimiensa kautta: sen ennuste on korkeimmillaan horisontin alussa (noin 2 780 askeleessa 1, noin 2 900 askeleessa 2), tasoittuu pohjalukemaan lähellä askelta 9 (noin 2 200) ja nousee jälleen askeleeseen 12 mennessä (noin 2 800). STEPAR-perustasoon verrattuna se on **150-660 korvausta korkeampi** suurimman osan horisonttia (katso `Winters - STEPAR` -sarake vaiheessa 4) - tuo ero on se kausisignaali, jonka autoregressiivinen malli jättää huomiotta.
- **95 %:n luottamusväli** (`L95_*`/`U95_*`-sarakkeet, joita ohjaa `ALPHA=`) levenee horisontin pidentyessä - WINTERS-mallilla noin 410 korvauksen leveydestä askeleessa 1 noin 1 420:een askeleessa 12 - rehellinen signaali siitä, että myöhäisen horisontin arviot kantavat enemmän epävarmuutta kuin lähiajan arviot. Varaustenlaskijoiden tulisi varata pääomaa ylärajaa vasten, ei pelkkää pisteennustetta vasten.
- **BY-käsittely** tuottaa Auto- ja Koti-ennusteet yhdellä ajolla, kummallakin oma kausisovituksensa. Autolinja ennustaa noin 2 510-2 600 välillä, kun taas kotilinja on lähellä 1 870-2 030, joten kumpikin linja säilyttää oman tasonsa ja kausimuotonsa - mallin, jota ryhmä laajentaisi jokaiseen tuotteeseen salkussa.

**Yhteenveto:** korvausmääräsarjalle, jolla on selkeä vuosisykli, `METHOD=WINTERS SEASONS=12` on idiomaattinen valinta; STEPAR-perustaso on hyödyllinen järkevyystarkistus, ja `OUTFULL`/`OUTLIMIT` yhdessä askel askeleelta -mallivertailun kanssa antavat aktuaarille mahdollisuuden mitoittaa eteenpäin suuntautuva varaus epävarmuusvyöhykkeineen.

> **Moottorin tarkkuushuomautus.** Tämä muistikirja dokumentoi kaksi nykyistä Jenner PROC FORECAST -rajoitusta (puutetesti `400979`): ennustehorisontin `ID`-arvoa edistetään yhdellä yksiköllä per askel eikä `INTERVAL=MONTH`-asetuksen mukaan, joten tulostetut ennustepäivämäärät eivät ole tarkoitetut vuoden 2025 kalenterikuukaudet (tarkastelemme horisonttia askeleen mukaan sen sijaan); ja `OUTRESID`/`OUTALL` eivät vielä täytä yhden askeleen `_TYPE_='RESIDUAL'`-rivejä, joten residuaalidiagnostiikka korvataan suoralla kahden mallin vertailulla. Itse pisteennusteet ja luottamusvälit kuitenkin tuotetaan, ja niihin yllä oleva kertomus perustuu.